<a href="https://colab.research.google.com/github/hamsikee/claw-code/blob/main/Colab_Inference(ram%20%EA%B3%B5%EA%B0%84%20%EA%B3%A0%EB%A0%A4%ED%95%98%EC%A7%80%EC%95%8A%EC%9D%8C).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INHA AI Challenge - Colab 추론 전용

제공된 `baseline_diffusion.ckpt`로 mp4 216개를 생성하고 제출 CSV를 만든다. **학습은 하지 않는다** (Colab 세션 제한 때문에 불가능하므로 학습은 GPU 서버 몫).

## 설계 의도

| 항목 | 위치 | 이유 |
|---|---|---|
| `open.zip` | Drive | 세션마다 재업로드하지 않으려고 |
| 압축 해제본 | `/content/open` | Drive는 15GB뿐, `/content`는 ~100GB |
| `backbone.ckpt` (10GB) | `/content` | Drive 용량 초과 방지. 세션마다 재다운로드 (5~15분) |
| **생성 mp4 + 제출 CSV** | **Drive** | **세션이 끊겨도 결과가 살아남는다** |

## 세션이 끊겼을 때

`generate_baseline_videos.py:90`에 `if not args.overwrite and all(path.exists() ...): continue` 가 있어 **이미 만들어진 mp4는 건너뛴다.** 따라서 재접속 후 2~8번 셀을 다시 실행하면 중단된 지점부터 이어서 생성된다.

**런타임 > 런타임 유형 변경 > GPU(T4)** 를 먼저 선택할 것.

## 1. GPU 확인

In [1]:
!nvidia-smi

import torch

print('torch    :', torch.__version__)
print('available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('GPU가 없습니다. [런타임 > 런타임 유형 변경 > T4 GPU] 를 선택하세요.')
print('device   :', torch.cuda.get_device_name(0))
print('VRAM     :', round(torch.cuda.mem_get_info()[1] / 1e9, 1), 'GB')

Wed Aug  5 07:52:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Drive 마운트 및 open.zip 찾기

In [2]:
from google.colab import drive

drive.mount('/content/drive')

# open.zip 위치 확인 (9.2GB 이상인 파일만)
!find '/content/drive/MyDrive' -iname 'open.zip' -size +1G 2>/dev/null

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/open.zip


## 3. 압축 해제

`INFERENCE_ONLY = True`면 `data/train`(8.4GB, 파일 23,473개)을 건너뛴다. 추론에는 `so100_action_statistics.json` 하나만 필요하다. 압축 해제 시간이 수 분 단축된다.

In [3]:
import zipfile
from pathlib import Path

# 2번 셀의 find 결과에 맞춰 수정할 것
ZIP_PATH = '/content/drive/MyDrive/Colab Notebooks/open.zip'
EXTRACT_TO = Path('/content/open')
INFERENCE_ONLY = True

assert Path(ZIP_PATH).exists(), f'zip을 찾을 수 없습니다: {ZIP_PATH}'

if (EXTRACT_TO / 'baseline' / 'challenge_kit').is_dir():
    print('이미 압축 해제되어 있습니다:', EXTRACT_TO)
else:
    with zipfile.ZipFile(ZIP_PATH) as z:
        names = z.namelist()
        if INFERENCE_ONLY:
            keep = [
                n for n in names
                if not n.startswith('data/train/')
                or n == 'data/train/so100_action_statistics.json'
            ]
            print(f'선택 해제: {len(keep)}/{len(names)} 개 (data/train 제외)')
        else:
            keep = names
        z.extractall(EXTRACT_TO, members=keep)
    print('압축 해제 완료:', EXTRACT_TO)

!du -sh /content/open

선택 해제: 544/24016 개 (data/train 제외)
압축 해제 완료: /content/open
216M	/content/open


## 4. 경로 고정 (절대경로)

런타임이 재시작되면 `cwd`가 `/content`로 돌아가 상대경로가 전부 깨진다. 이 셀은 재시작 후에도 항상 다시 실행할 것.

In [4]:
import os
from pathlib import Path

BASELINE_ROOT   = Path('/content/open/baseline')
PACKAGE_ROOT    = BASELINE_ROOT.parent
KIT_ROOT        = BASELINE_ROOT / 'challenge_kit'
DATA_ROOT       = PACKAGE_ROOT / 'data'
SUBMISSION_KIT  = PACKAGE_ROOT / 'submission_kit'
ACTION_STATS    = DATA_ROOT / 'train/so100_action_statistics.json'

# 결과물은 Drive에 저장해야 세션이 끊겨도 남는다
DRIVE_OUT = Path('/content/drive/MyDrive/inha_out')
PRED_DIR  = DRIVE_OUT / 'input_videos'
PRED_DIR.mkdir(parents=True, exist_ok=True)

assert KIT_ROOT.is_dir(), '3번 셀(압축 해제)을 먼저 실행하세요.'
os.chdir(BASELINE_ROOT)

print('cwd         :', os.getcwd())
print('eval images :', len(list((DATA_ROOT / 'eval/images').glob('*.png'))), '(216이어야 정상)')
print('eval actions:', len(list((DATA_ROOT / 'eval/actions').glob('*.npy'))), '(216이어야 정상)')
print('action stats:', ACTION_STATS.exists())
print('baseline ckpt:', (BASELINE_ROOT / 'checkpoints/baseline_diffusion.ckpt').exists())
print('출력 폴더    :', PRED_DIR)
print('생성된 mp4  :', len(list(PRED_DIR.glob('*.mp4'))), '/ 216')

cwd         : /content/open/baseline
eval images : 216 (216이어야 정상)
eval actions: 216 (216이어야 정상)
action stats: True
baseline ckpt: True
출력 폴더    : /content/drive/MyDrive/inha_out/input_videos
생성된 mp4  : 0 / 216


## 5. 패키지 설치

⚠️ `requirements.txt`가 `torch==2.7.1`을 고정하므로 Colab 기본 torch가 교체된다. **설치 후 런타임 재시작이 필요할 수 있다.** 재시작하면 1·4번 셀만 다시 실행하고 (3·5번은 건너뛰고) 6번으로 진행하면 된다.

In [5]:
import subprocess
import sys

print('Python:', sys.version.split()[0])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', str(BASELINE_ROOT / 'requirements.txt')])

# setup.py는 아래 3곳에만 있다. baseline/ 자체에는 없으므로 '-e .' 는 실패한다.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                       '-e', str(KIT_ROOT),
                       '-e', str(KIT_ROOT / 'libs/dynamicrafter'),
                       '-e', str(BASELINE_ROOT / 'shared_libs/video_utils')])

# wandb는 이 프로젝트에서 쓰지 않으며 protobuf 버전 충돌을 일으킨다
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'wandb'])

check = subprocess.run(
    [sys.executable, '-c',
     'import torch, lvdm, ldwma; print("import OK | torch", torch.__version__, "| cuda", torch.cuda.is_available())'],
    capture_output=True, text=True,
)
print(check.stdout or check.stderr)

Python: 3.12.13
import OK | torch 2.7.1+cu126 | cuda True



## 6. backbone 다운로드 (~10GB)

제공된 `baseline_diffusion.ckpt`는 UNet 가중치만 담고 있다. VAE / CLIP / image_proj는 이 backbone에서 로드하므로 **추론에도 반드시 필요하다**.

`wget -c`는 중단 시 이어받기가 된다 (원본 노트북의 `urlretrieve`는 안 됨).

In [6]:
backbone = BASELINE_ROOT / 'checkpoints/backbone.ckpt'
backbone.parent.mkdir(parents=True, exist_ok=True)

url = 'https://huggingface.co/Doubiiu/DynamiCrafter_512/resolve/main/model.ckpt'
!wget -c -O "{backbone}" "{url}"

!ls -lh "{backbone}"
!df -h /content | tail -1

--2026-08-05 07:57:20--  https://huggingface.co/Doubiiu/DynamiCrafter_512/resolve/main/model.ckpt
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.17, 18.164.174.23, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/65bde79db54ab5b37de8d0f5/75b528fa59900c3d1f936d4271cb49c3103013a693e7e359eedf51e61308d04b?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.ckpt%3B+filename%3D%22model.ckpt%22%3B&X-Xet-Cas-Uid=public&Expires=1785920240&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjViZGU3OWRiNTRhYjViMzdkZThkMGY1Lzc1YjUyOGZhNTk5MDBjM2QxZjkzNmQ0MjcxY2I0OWMzMTAzMDEzYTY5M2U3ZTM1OWVlZGY1MWU2MTMwOGQwNGJcXD91c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitmaWxlbmFtZSUyQSUzRFVURi04JTI3JTI3bW9kZWwuY2twdCUzQitmaWxlbmFtZSUzRCUyMm1vZGVsLmNrcHQlMjI

In [15]:
import os
for p in ['checkpoints/backbone.ckpt', 'checkpoints/baseline_diffusion.ckpt']:
    f = '/content/open/baseline/' + p
    print(os.path.exists(f), round(os.path.getsize(f)/1e6, 1) if os.path.exists(f) else '-', p)


!ls /content/open/data/eval/images/*.png | wc -l
!ls /content/open/data/eval/actions/*.npy | wc -l
!python -c "import torch,lvdm,ldwma,open_clip;print('OK',torch.__version__,torch.cuda.is_available())"


%cd /content/open/baseline/challenge_kit

!python scripts/inference/generate_baseline_videos.py --checkpoint ../checkpoints/baseline_diffusion.ckpt --challenge-root ../../data/eval --prediction-root /content/drive/MyDrive/inha_out/input_videos --action-stats-path ../../data/train/so100_action_statistics.json

True 10437.5 checkpoints/backbone.ckpt
True 87.7 checkpoints/baseline_diffusion.ckpt
216
216
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
OK 2.7.1+cu126 True
/content/open/baseline/challenge_kit
AE working on z of shape (1, 4, 32, 32) = 4096 dimensions.
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
^C


## 7. 영상 생성 (216개)

T4 기준 2~4시간 소요. **세션이 끊기면 1·4번 셀 실행 후 이 셀만 다시 실행하면 이어서 진행된다.**

속도를 올리고 싶으면 `--ddim-steps`를 낮출 수 있으나(기본 50), 생성 품질이 떨어져 점수에 영향을 준다. VRAM에 여유가 있으면 `--batch-size 2`가 더 안전한 선택이다.

In [11]:
import os
import subprocess
import sys

env = os.environ.copy()
env['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

cmd = [
    sys.executable, 'scripts/inference/generate_baseline_videos.py',
    '--checkpoint',        str(BASELINE_ROOT / 'checkpoints/baseline_diffusion.ckpt'),
    '--challenge-root',    str(DATA_ROOT / 'eval'),
    '--prediction-root',   str(PRED_DIR),
    '--action-stats-path', str(ACTION_STATS),
    '--batch-size',        '1',
]
print('cwd:', KIT_ROOT)
print('cmd:', ' '.join(cmd), '\n')

proc = subprocess.Popen(cmd, cwd=str(KIT_ROOT), env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()

print('\n=== returncode:', rc, '===')
print('생성된 mp4:', len(list(PRED_DIR.glob('*.mp4'))), '/ 216')

cwd: /content/open/baseline/challenge_kit
cmd: /usr/bin/python3 scripts/inference/generate_baseline_videos.py --checkpoint /content/open/baseline/checkpoints/baseline_diffusion.ckpt --challenge-root /content/open/data/eval --prediction-root /content/drive/MyDrive/inha_out/input_videos --action-stats-path /content/open/data/train/so100_action_statistics.json --batch-size 1 

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

=== returncode: -9 ===
생성된 mp4: 0 / 216


## 8. 진행 상황 확인

216개가 아니면 7번 셀을 다시 실행할 것.

In [8]:
mp4s = sorted(PRED_DIR.glob('*.mp4'))
print(f'{len(mp4s)} / 216')

expected = {f'sample_{i:06d}' for i in range(216)}
missing = sorted(expected - {p.stem for p in mp4s})
if missing:
    print(f'누락 {len(missing)}개, 예: {missing[:10]}')
else:
    print('전부 생성 완료. 9번 셀로 진행하세요.')
    total_mb = sum(p.stat().st_size for p in mp4s) / 1e6
    print(f'총 용량: {total_mb:.1f} MB')

0 / 216
누락 216개, 예: ['sample_000000', 'sample_000001', 'sample_000002', 'sample_000003', 'sample_000004', 'sample_000005', 'sample_000006', 'sample_000007', 'sample_000008', 'sample_000009']


## 9. 제출 CSV 생성

mp4가 216개 미만이면 `make_submission_csv.py`가 `FileNotFoundError`로 중단시킨다 (스크립트가 알아서 막아준다).

In [9]:
import subprocess
import sys

# submission_kit은 baseline과 의존성이 별개다
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', str(SUBMISSION_KIT / 'requirements.txt')])

OUTPUT_CSV = DRIVE_OUT / 'submission_features.csv'

cmd = [
    sys.executable, 'make_submission_csv.py',
    '--prediction-root',   str(PRED_DIR),
    '--challenge-root',    str(DATA_ROOT / 'eval'),
    '--action-stats-path', str(ACTION_STATS),
    '--output-csv',        str(OUTPUT_CSV),
]
print('cmd:', ' '.join(cmd), '\n')

proc = subprocess.Popen(cmd, cwd=str(SUBMISSION_KIT),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
print('\n=== returncode:', proc.wait(), '===')

cmd: /usr/bin/python3 make_submission_csv.py --prediction-root /content/drive/MyDrive/inha_out/input_videos --challenge-root /content/open/data/eval --action-stats-path /content/open/data/train/so100_action_statistics.json --output-csv /content/drive/MyDrive/inha_out/submission_features.csv 

Traceback (most recent call last):
  File "/content/open/submission_kit/make_submission_csv.py", line 74, in <module>
    main()
  File "/content/open/submission_kit/make_submission_csv.py", line 44, in main
    raise FileNotFoundError(
FileNotFoundError: /content/drive/MyDrive/inha_out/input_videos에 제출 변환 대상 mp4가 부족합니다. 총 216개 누락. 예: sample_000000.mp4, sample_000001.mp4, sample_000002.mp4, sample_000003.mp4, sample_000004.mp4, sample_000005.mp4, sample_000006.mp4, sample_000007.mp4, sample_000008.mp4, sample_000009.mp4

=== returncode: 1 ===


## 10. 제출 파일 검증

`(648, 3)` 이고 3개 컴포넌트가 각각 216개여야 정상이다.

In [10]:
import pandas as pd

OUTPUT_CSV = DRIVE_OUT / 'submission_features.csv'
submission = pd.read_csv(OUTPUT_CSV)

print('shape:', submission.shape)
display(submission.head())
print(submission['feature_component'].value_counts())

assert submission.shape == (648, 3), f'행/열 수가 다릅니다: {submission.shape}'
assert set(submission['feature_component'].value_counts()) == {216}, '컴포넌트별 개수가 216이 아닙니다'
assert submission['sample_id'].nunique() == 216, 'sample_id가 216개가 아닙니다'

size_mb = OUTPUT_CSV.stat().st_size / 1e6
print(f'\n검증 통과. {OUTPUT_CSV} ({size_mb:.1f} MB)')
print('Drive에 저장되어 있으므로 세션이 끊겨도 안전합니다. 이 파일을 데이콘에 제출하세요.')

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 11. (선택) 로컬로 내려받기

In [ ]:
from google.colab import files

files.download(str(DRIVE_OUT / 'submission_features.csv'))